In [2]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from torch_geometric.nn import GCNConv
from pathlib import Path
from sklearn.metrics import mean_squared_error

# ==========================================
# 1. Load the PyG Sequence
# ==========================================
PROCESSED_DIR = Path("../data/processed")
graph_snapshots = torch.load(PROCESSED_DIR / "temporal_graph_sequence.pt", weights_only=False)
node_index = pd.read_csv(PROCESSED_DIR / "bis_cbs_node_index.csv")
NUM_NODES = len(node_index)

# Rebuild Node Features (In/Out Degree and Strength)
for graph in graph_snapshots:
    src, dst = graph.edge_index
    weights = graph.edge_attr.view(-1)
    x = torch.zeros((NUM_NODES, 4), dtype=torch.float)
    x[:, 0].scatter_add_(0, src, weights) # Out-strength
    x[:, 1].scatter_add_(0, dst, weights) # In-strength
    x[:, 2].scatter_add_(0, src, torch.ones_like(src, dtype=torch.float)) # Out-degree
    x[:, 3].scatter_add_(0, dst, torch.ones_like(dst, dtype=torch.float)) # In-degree
    
    # Clamp values to prevent negative logarithms (NaNs)
    graph.x = torch.log1p(torch.clamp(x, min=0.0))
    graph.num_nodes = NUM_NODES

# ==========================================
# 2. Build Spatio-Temporal GNN (GCN + LSTM)
# ==========================================
class TemporalGNN(nn.Module):
    def __init__(self, node_features, hidden_dim):
        super().__init__()
        self.gcn1 = GCNConv(node_features, hidden_dim)
        self.gcn2 = GCNConv(hidden_dim, hidden_dim)
        self.lstm = nn.LSTM(input_size=hidden_dim, hidden_size=hidden_dim, batch_first=True)
        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, window_graphs, target_edge_index):
        temporal_embeddings = []
        for g in window_graphs:
            h = self.gcn1(g.x, g.edge_index).relu()
            h = self.gcn2(h, g.edge_index).relu()
            temporal_embeddings.append(h)
            
        seq_tensor = torch.stack(temporal_embeddings, dim=1) 
        lstm_out, (h_n, c_n) = self.lstm(seq_tensor)
        final_node_embeddings = lstm_out[:, -1, :] 
        
        src, dst = target_edge_index
        src_emb = final_node_embeddings[src]
        dst_emb = final_node_embeddings[dst]
        
        edge_features = torch.cat([src_emb, dst_emb], dim=1)
        predictions = self.edge_mlp(edge_features)
        
        return predictions

# ==========================================
# 3. Rolling Window Training Setup
# ==========================================
WINDOW_SIZE = 4 # Look back 4 quarters
HIDDEN_DIM = 32
model = TemporalGNN(node_features=4, hidden_dim=HIDDEN_DIM)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
criterion = nn.MSELoss()

train_graphs = [g for g in graph_snapshots if int(g.period[:4]) <= 2018]
val_graphs = [g for g in graph_snapshots if 2019 <= int(g.period[:4]) <= 2021]
test_graphs = [g for g in graph_snapshots if int(g.period[:4]) >= 2022]

print(f"Training ST-GNN on rolling window of {WINDOW_SIZE} quarters...")

# ==========================================
# 4. Training Loop
# ==========================================
EPOCHS = 50
model.train()

for epoch in range(1, EPOCHS + 1):
    epoch_loss = 0.0
    for t in range(len(train_graphs) - WINDOW_SIZE):
        window = train_graphs[t : t + WINDOW_SIZE]
        target_graph = train_graphs[t + WINDOW_SIZE]
        
        optimizer.zero_grad()
        preds = model(window, target_graph.edge_index)
        loss = criterion(preds, target_graph.edge_attr)
        loss.backward()
        
        # THE FIX: Clip gradients to prevent LSTM explosions
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        epoch_loss += loss.item()
        
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d} | Train MSE: {epoch_loss / (len(train_graphs) - WINDOW_SIZE):.4f}")

# ==========================================
# 5. Out-of-Sample Forecasting (Test Set)
# ==========================================
model.eval()
test_predictions = []
test_actuals = []

with torch.no_grad():
    context_graphs = val_graphs[-WINDOW_SIZE:] + test_graphs
    
    for t in range(len(test_graphs)):
        window = context_graphs[t : t + WINDOW_SIZE]
        target_graph = context_graphs[t + WINDOW_SIZE]
        
        preds = model(window, target_graph.edge_index)
        
        test_predictions.extend(preds.view(-1).tolist())
        test_actuals.extend(target_graph.edge_attr.view(-1).tolist())

mse_test = mean_squared_error(test_actuals, test_predictions)
print(f"\nOut-of-Sample Test MSE (2022-2026): {mse_test:.4f}")

# Save the ST-GNN model
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(exist_ok=True)
torch.save(model.state_dict(), MODELS_DIR / "st_gnn_model.pth")
print(f"Saved temporal predictive model to {MODELS_DIR}")

Training ST-GNN on rolling window of 4 quarters...
Epoch 001 | Train MSE: 0.6915
Epoch 010 | Train MSE: 0.7003
Epoch 020 | Train MSE: 0.6115
Epoch 030 | Train MSE: 0.5541
Epoch 040 | Train MSE: 0.5544
Epoch 050 | Train MSE: 0.5508

Out-of-Sample Test MSE (2022-2026): 0.5981
Saved temporal predictive model to ..\models
